# Task 3: Learning to Rank on WikIR Data

In [1]:
# Cell 1: Load data
import pandas as pd
import numpy as np
from collections import Counter
import math
import re

DATA = 'wikIR1k-2'

# documents
docs_df = pd.read_csv(f'{DATA}/documents.csv')
docs_df = docs_df.set_index('id_right')
docs = docs_df['text_right'].to_dict()

# queries
train_queries_df = pd.read_csv(f'{DATA}/training/queries.csv')
test_queries_df  = pd.read_csv(f'{DATA}/test/queries.csv')
train_queries = dict(zip(train_queries_df['id_left'], train_queries_df['text_left']))
test_queries  = dict(zip(test_queries_df['id_left'],  test_queries_df['text_left']))

# real qrels (TREC format: qid 0 doc_id relevance)
def load_trec_qrels(path):
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                rows.append({'id_left': int(parts[0]), 'id_right': int(parts[2]), 'label': int(parts[3])})
    return pd.DataFrame(rows)

train_qrels_real = load_trec_qrels(f'{DATA}/training/qrels')
# BM25.qrels.csv still used for Cell 4 feature verification example
train_qrels = pd.read_csv(f'{DATA}/training/BM25.qrels.csv')[['id_left','id_right','label']]

# BM25 top-100 results for training (used to sample non-relevant docs)
train_bm25 = pd.read_csv(f'{DATA}/training/BM25.res', sep=' ',
                          names=['qid','Q0','doc_id','rank','score','method'])

# BM25 results for test (top100 per query)
test_bm25 = pd.read_csv(f'{DATA}/test/BM25.res', sep=' ',
                         names=['qid','Q0','doc_id','rank','score','method'])

print(f'Documents:            {len(docs)}')
print(f'Train queries:        {len(train_queries)}')
print(f'Test queries:         {len(test_queries)}')
print(f'Train qrels (real):   {len(train_qrels_real)}')
print(f'Train BM25 rows:      {len(train_bm25)}')
print(f'Test BM25 rows:       {len(test_bm25)}')


Documents:            369721
Train queries:        1444
Test queries:         100
Train qrels (real):   47699
Train BM25 rows:      144400
Test BM25 rows:       10000


In [2]:
# Cell 2: Preprocess - tokenize
def tokenize(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\b[a-z]+\b', text.lower())

# precompute tokenized docs and queries
print('Tokenizing documents...')
tok_docs = {doc_id: tokenize(text) for doc_id, text in docs.items()}

print('Tokenizing queries...')
tok_train_queries = {qid: tokenize(text) for qid, text in train_queries.items()}
tok_test_queries  = {qid: tokenize(text) for qid, text in test_queries.items()}

# compute document frequencies for IDF
print('Computing IDF...')
N = len(tok_docs)
df_counts = Counter()
for tokens in tok_docs.values():
    for t in set(tokens):
        df_counts[t] += 1

def idf(term):
    return math.log((N - df_counts[term] + 0.5) / (df_counts[term] + 0.5) + 1)

# average doc length for BM25
doc_lengths = {doc_id: len(tokens) for doc_id, tokens in tok_docs.items()}
avg_dl = np.mean(list(doc_lengths.values()))
print(f'Avg doc length: {avg_dl:.1f} tokens')
print('Preprocessing done.')

Tokenizing documents...
Tokenizing queries...
Computing IDF...
Avg doc length: 189.4 tokens
Preprocessing done.


In [3]:
# Cell 3: Feature extraction
# Features for a query-document pair:
#  1.  query_length          - number of words in query
#  2.  doc_length            - number of words in document
#  3.  num_matched_terms     - number of unique query terms found in doc
#  4.  frac_matched_terms    - fraction of query terms found in doc
#  5.  sum_tf                - sum of term frequencies of query terms in doc
#  6.  max_tf                - max TF of any query term in doc
#  7.  avg_tf                - average TF of query terms in doc
#  8.  sum_idf               - sum of IDF of query terms
#  9.  avg_idf               - average IDF of query terms
# 10.  sum_tfidf             - sum of TF*IDF of query terms in doc
# 11.  bm25_score            - BM25 score (k1=1.5, b=0.75)
# 12.  min_span              - smallest window in doc covering all matched query terms
# 13.  first_match_pos       - position of first matched query term in doc (normalized)
# 14.  query_doc_length_ratio- query length / doc length

K1 = 1.5
B  = 0.75

def bm25_score(query_terms, doc_tokens, doc_len):
    tf_map = Counter(doc_tokens)
    score = 0.0
    for term in set(query_terms):
        tf = tf_map.get(term, 0)
        if tf == 0:
            continue
        idf_val = idf(term)
        tf_norm = (tf * (K1 + 1)) / (tf + K1 * (1 - B + B * doc_len / avg_dl))
        score += idf_val * tf_norm
    return score

def min_span_window(query_terms, doc_tokens):
    """Minimum span (in tokens) covering all matched query terms."""
    matched = set(query_terms) & set(doc_tokens)
    if len(matched) < 2:
        return len(doc_tokens)  # no span possible
    positions = {term: [] for term in matched}
    for i, tok in enumerate(doc_tokens):
        if tok in positions:
            positions[tok].append(i)
    # sliding window approach
    all_pos = sorted((pos, term) for term, pos_list in positions.items() for pos in pos_list)
    term_count = Counter()
    covered = 0
    left = 0
    min_span = len(doc_tokens)
    for right, (pos, term) in enumerate(all_pos):
        term_count[term] += 1
        if term_count[term] == 1:
            covered += 1
        while covered == len(matched):
            span = all_pos[right][0] - all_pos[left][0] + 1
            min_span = min(min_span, span)
            left_term = all_pos[left][1]
            term_count[left_term] -= 1
            if term_count[left_term] == 0:
                covered -= 1
            left += 1
    return min_span

def extract_features(query_tokens, doc_tokens, doc_len):
    tf_map = Counter(doc_tokens)
    query_terms = set(query_tokens)
    matched = query_terms & set(doc_tokens)

    # basic counts
    query_length       = len(query_tokens)
    doc_length         = doc_len
    num_matched        = len(matched)
    frac_matched       = num_matched / max(len(query_terms), 1)

    # TF features
    tfs = [tf_map.get(t, 0) for t in query_terms]
    sum_tf  = sum(tfs)
    max_tf  = max(tfs) if tfs else 0
    avg_tf  = np.mean(tfs) if tfs else 0

    # IDF features
    idfs = [idf(t) for t in query_terms]
    sum_idf = sum(idfs)
    avg_idf = np.mean(idfs) if idfs else 0

    # TF-IDF
    sum_tfidf = sum(tf_map.get(t, 0) * idf(t) for t in query_terms)

    # BM25
    bm25 = bm25_score(query_tokens, doc_tokens, doc_len)

    # min span
    span = min_span_window(list(query_terms), doc_tokens)
    norm_span = span / max(doc_len, 1)

    # first match position (normalized)
    first_pos = doc_len  # default: not found
    for i, tok in enumerate(doc_tokens):
        if tok in query_terms:
            first_pos = i
            break
    norm_first_pos = first_pos / max(doc_len, 1)

    # ratio
    ql_dl_ratio = query_length / max(doc_len, 1)

    return [
        query_length, doc_length, num_matched, frac_matched,
        sum_tf, max_tf, avg_tf,
        sum_idf, avg_idf, sum_tfidf,
        bm25, norm_span, norm_first_pos, ql_dl_ratio
    ]

FEATURE_NAMES = [
    'query_length', 'doc_length', 'num_matched_terms', 'frac_matched_terms',
    'sum_tf', 'max_tf', 'avg_tf',
    'sum_idf', 'avg_idf', 'sum_tfidf',
    'bm25_score', 'norm_min_span', 'norm_first_match_pos', 'query_doc_length_ratio'
]

print(f'Feature set ({len(FEATURE_NAMES)} features):')
for i, f in enumerate(FEATURE_NAMES, 1):
    print(f'  {i:2d}. {f}')

Feature set (14 features):
   1. query_length
   2. doc_length
   3. num_matched_terms
   4. frac_matched_terms
   5. sum_tf
   6. max_tf
   7. avg_tf
   8. sum_idf
   9. avg_idf
  10. sum_tfidf
  11. bm25_score
  12. norm_min_span
  13. norm_first_match_pos
  14. query_doc_length_ratio


In [4]:
# Cell 4: Verify features on one example
sample_qid  = list(tok_train_queries.keys())[0]
sample_qtok = tok_train_queries[sample_qid]
sample_qrels_row = train_qrels[train_qrels['id_left'] == sample_qid].iloc[0]
sample_did  = sample_qrels_row['id_right']
sample_dtok = tok_docs.get(sample_did, [])

feats = extract_features(sample_qtok, sample_dtok, len(sample_dtok))

print(f'Query: "{train_queries[sample_qid]}" (tokens: {sample_qtok})')
print(f'Doc ID: {sample_did}, length: {len(sample_dtok)} tokens')
print(f'Label: {sample_qrels_row["label"]}')
print()
for name, val in zip(FEATURE_NAMES, feats):
    print(f'  {name:30s}: {val:.4f}')

Query: "yanni" (tokens: ['yanni'])
Doc ID: 806300, length: 191 tokens
Label: 1

  query_length                  : 1.0000
  doc_length                    : 191.0000
  num_matched_terms             : 1.0000
  frac_matched_terms            : 1.0000
  sum_tf                        : 6.0000
  max_tf                        : 6.0000
  avg_tf                        : 6.0000
  sum_idf                       : 10.3782
  avg_idf                       : 10.3782
  sum_tfidf                     : 62.2690
  bm25_score                    : 20.7295
  norm_min_span                 : 1.0000
  norm_first_match_pos          : 0.0995
  query_doc_length_ratio        : 0.0052


In [5]:
# Cell 5: Build training vectors
# Relevant docs: from real training qrels (labels 1 and 2)
# Non-relevant docs: BM25 top-100 docs NOT in real qrels for that query (label=0)
# Equal number of relevant and non-relevant sampled per query
import random
random.seed(42)

# build set of relevant doc IDs per query from real qrels
real_relevant = train_qrels_real.groupby('id_left')['id_right'].apply(set).to_dict()

# build BM25 top-100 doc list per query for sampling non-relevant
train_bm25_docs = train_bm25.groupby('qid')['doc_id'].apply(list).to_dict()

X_train, y_train, qids_train = [], [], []
train_query_ids = list(train_queries.keys())
skipped = 0

for qid in train_query_ids:
    rel_docs = real_relevant.get(qid, set())
    if not rel_docs:
        skipped += 1
        continue

    q_tokens = tok_train_queries.get(qid, [])
    if not q_tokens:
        skipped += 1
        continue

    # non-relevant: BM25 top-100 docs not in real qrels
    bm25_docs = train_bm25_docs.get(qid, [])
    nonrel_docs = [d for d in bm25_docs if d not in rel_docs]
    if not nonrel_docs:
        skipped += 1
        continue

    # relevant rows with labels
    rel_rows = train_qrels_real[train_qrels_real['id_left'] == qid]

    # sample equal number of non-relevant
    n_sample = min(len(rel_docs), len(nonrel_docs))
    sampled_nonrel = random.sample(nonrel_docs, n_sample)

    for _, row in rel_rows.iterrows():
        did = row['id_right']
        d_tokens = tok_docs.get(did, [])
        feats = extract_features(q_tokens, d_tokens, len(d_tokens))
        X_train.append(feats)
        y_train.append(int(row['label']))
        qids_train.append(str(qid))

    for did in sampled_nonrel:
        d_tokens = tok_docs.get(did, [])
        feats = extract_features(q_tokens, d_tokens, len(d_tokens))
        X_train.append(feats)
        y_train.append(0)
        qids_train.append(str(qid))

X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train)
qids_train = np.array(qids_train)

print(f'Training pairs:  {len(X_train)}')
print(f'Queries used:    {len(set(qids_train))}')
print(f'Queries skipped: {skipped}')
print(f'Label dist: {dict(zip(*np.unique(y_train, return_counts=True)))}')


Training pairs:  73846
Queries used:    1436
Queries skipped: 8
Label dist: {np.int64(0): np.int64(26252), np.int64(1): np.int64(46158), np.int64(2): np.int64(1436)}


In [6]:
# Cell 6: Train ranking function - LambdaMart with NDCG
# Choice motivation:
# - LambdaMart directly optimizes NDCG (our target metric)
# - Outperformed YetiRank in Task 2.2 experiments
# - Groupwise method: works with query groups and graded relevance labels
# Target metric: NDCG - standard IR metric, position-aware, handles graded relevance

from catboost import CatBoostRanker, Pool

train_pool = Pool(data=X_train, label=y_train, group_id=qids_train)

model = CatBoostRanker(
    loss_function='LambdaMart:metric=NDCG',
    eval_metric='NDCG',
    iterations=500,
    learning_rate=0.1,
    random_seed=0,
    verbose=100,
)
model.fit(train_pool)
print('Training done.')

0:	total: 91.8ms	remaining: 45.8s
100:	total: 3.67s	remaining: 14.5s
200:	total: 7.4s	remaining: 11s
300:	total: 11.2s	remaining: 7.38s
400:	total: 15s	remaining: 3.69s
499:	total: 18.7s	remaining: 0us
Training done.


In [7]:
# Cell 7: Generate test vectors for top100 BM25 docs + re-rank + evaluate
import ir_measures
from ir_measures import nDCG, MAP, P
import os

os.makedirs('runs', exist_ok=True)

# build test feature matrix from top100 BM25 docs per test query
X_test, qids_test, doc_ids_test = [], [], []

for qid, group in test_bm25.groupby('qid'):
    q_tokens = tok_test_queries.get(qid, [])
    for _, row in group.iterrows():
        did = row['doc_id']
        d_tokens = tok_docs.get(did, [])
        feats = extract_features(q_tokens, d_tokens, len(d_tokens))
        X_test.append(feats)
        qids_test.append(str(qid))
        doc_ids_test.append(did)

X_test = np.array(X_test, dtype=np.float32)
qids_test = np.array(qids_test)

# predict and save TREC run
scores = model.predict(X_test)
results = pd.DataFrame({'qid': qids_test, 'doc_id': doc_ids_test, 'score': scores})

with open('runs/lambdamart_14feat.res', 'w') as f:
    for qid, group in results.groupby('qid'):
        ranked = group.sort_values('score', ascending=False).reset_index(drop=True)
        for rank, row in ranked.iterrows():
            f.write(f"{qid} Q0 {row['doc_id']} {rank+1} {row['score']:.6f} LambdaMart14\n")

# evaluate with ir_measures against real test/qrels
qrels_path = f'{DATA}/test/qrels'
measures = [P@10, MAP, nDCG@20]

qrels_ir = ir_measures.read_trec_qrels(qrels_path)
run_ir   = ir_measures.read_trec_run('runs/lambdamart_14feat.res')
agg = ir_measures.calc_aggregate(measures, qrels_ir, run_ir)

with open(f'{DATA}/test/BM25.metrics.json') as f:
    import json as _json
    bm25_metrics = _json.load(f)

print(f'=== 3.1 Results (vs Assignment 2 baseline) ===')
print(f'{"Method":<35} {"P@10":>6} {"MAP":>8} {"nDCG@20":>9}')
print('-' * 63)
print(f'{"BM25_original (Assignment 2)":<35} {0.212:>6.4f} {0.1752:>8.4f} {0.3570:>9.4f}')
print(f'{"LambdaMart (14 features)":<35} {agg[P@10]:>6.4f} {agg[MAP]:>8.4f} {agg[nDCG@20]:>9.4f}')


=== 3.1 Results (vs Assignment 2 baseline) ===
Method                                P@10      MAP   nDCG@20
---------------------------------------------------------------
BM25_original (Assignment 2)        0.2120   0.1752    0.3570
LambdaMart (14 features)            0.1480   0.1414    0.2678


# Task 3.2: Reconstruct BM25 Ranking
# Features: only the BM25 components — TF per query term, IDF per query term, doc length

In [8]:
# Cell 8: BM25 component features
# BM25 = sum over query terms of: IDF(t) * TF_norm(t, d)
# So we represent each query-doc pair with:
#   - TF of each query term in doc       (per-term, sum/max/avg)
#   - IDF of each query term             (per-term, sum/max/avg)
#   - document length
# This is the minimal set that fully determines BM25

def extract_bm25_features(query_tokens, doc_tokens, doc_len):
    tf_map = Counter(doc_tokens)
    query_terms = list(set(query_tokens))

    # per-term TF values
    tfs  = [tf_map.get(t, 0) for t in query_terms]
    idfs = [idf(t) for t in query_terms]

    sum_tf  = sum(tfs)
    max_tf  = max(tfs) if tfs else 0
    avg_tf  = np.mean(tfs) if tfs else 0

    sum_idf = sum(idfs)
    max_idf = max(idfs) if idfs else 0
    avg_idf = np.mean(idfs) if idfs else 0

    return [sum_tf, max_tf, avg_tf, sum_idf, max_idf, avg_idf, doc_len]

BM25_FEATURE_NAMES = ['sum_tf', 'max_tf', 'avg_tf', 'sum_idf', 'max_idf', 'avg_idf', 'doc_length']
print(f'BM25 component features ({len(BM25_FEATURE_NAMES)}): {BM25_FEATURE_NAMES}')

BM25 component features (7): ['sum_tf', 'max_tf', 'avg_tf', 'sum_idf', 'max_idf', 'avg_idf', 'doc_length']


In [9]:
# Cell 9: Build training vectors with BM25 features + train
X_train_bm25, y_train_bm25, qids_train_bm25 = [], [], []

for qid in train_query_ids:
    rel_docs = real_relevant.get(qid, set())
    if not rel_docs:
        continue

    q_tokens = tok_train_queries.get(qid, [])
    if not q_tokens:
        continue

    bm25_docs = train_bm25_docs.get(qid, [])
    nonrel_docs = [d for d in bm25_docs if d not in rel_docs]
    if not nonrel_docs:
        continue

    rel_rows = train_qrels_real[train_qrels_real['id_left'] == qid]
    n_sample = min(len(rel_docs), len(nonrel_docs))
    sampled_nonrel = random.sample(nonrel_docs, n_sample)

    for _, row in rel_rows.iterrows():
        did = row['id_right']
        d_tokens = tok_docs.get(did, [])
        feats = extract_bm25_features(q_tokens, d_tokens, len(d_tokens))
        X_train_bm25.append(feats)
        y_train_bm25.append(int(row['label']))
        qids_train_bm25.append(str(qid))

    for did in sampled_nonrel:
        d_tokens = tok_docs.get(did, [])
        feats = extract_bm25_features(q_tokens, d_tokens, len(d_tokens))
        X_train_bm25.append(feats)
        y_train_bm25.append(0)
        qids_train_bm25.append(str(qid))

X_train_bm25 = np.array(X_train_bm25, dtype=np.float32)
y_train_bm25 = np.array(y_train_bm25)
qids_train_bm25 = np.array(qids_train_bm25)

train_pool_bm25 = Pool(data=X_train_bm25, label=y_train_bm25, group_id=qids_train_bm25)

model_bm25 = CatBoostRanker(
    loss_function='LambdaMart:metric=NDCG',
    eval_metric='NDCG',
    iterations=500,
    learning_rate=0.1,
    random_seed=0,
    verbose=100,
)
model_bm25.fit(train_pool_bm25)
print('BM25-reconstruct model trained.')


0:	total: 28.8ms	remaining: 14.4s
100:	total: 3.41s	remaining: 13.5s
200:	total: 7.02s	remaining: 10.4s
300:	total: 10.7s	remaining: 7.08s
400:	total: 14.4s	remaining: 3.56s
499:	total: 18.1s	remaining: 0us
BM25-reconstruct model trained.


In [10]:
# Cell 10: Evaluate BM25-reconstruct model + final comparison
X_test_bm25, qids_test_bm25, doc_ids_test_bm25 = [], [], []

for qid, group in test_bm25.groupby('qid'):
    q_tokens = tok_test_queries.get(qid, [])
    for _, row in group.iterrows():
        did = row['doc_id']
        d_tokens = tok_docs.get(did, [])
        feats = extract_bm25_features(q_tokens, d_tokens, len(d_tokens))
        X_test_bm25.append(feats)
        qids_test_bm25.append(str(qid))
        doc_ids_test_bm25.append(did)

X_test_bm25 = np.array(X_test_bm25, dtype=np.float32)
scores_bm25 = model_bm25.predict(X_test_bm25)
results_bm25 = pd.DataFrame({'qid': qids_test_bm25, 'doc_id': doc_ids_test_bm25, 'score': scores_bm25})

with open('runs/lambdamart_bm25feat.res', 'w') as f:
    for qid, group in results_bm25.groupby('qid'):
        ranked = group.sort_values('score', ascending=False).reset_index(drop=True)
        for rank, row in ranked.iterrows():
            f.write(f"{qid} Q0 {row['doc_id']} {rank+1} {row['score']:.6f} LambdaMartBM25\n")

# evaluate with ir_measures against real test/qrels
qrels_ir2 = ir_measures.read_trec_qrels(qrels_path)
run_ir2   = ir_measures.read_trec_run('runs/lambdamart_bm25feat.res')
agg2 = ir_measures.calc_aggregate(measures, qrels_ir2, run_ir2)

print(f'=== Final Comparison (all methods vs Assignment 2 baseline) ===')
print(f'{"Method":<35} {"P@10":>6} {"MAP":>8} {"nDCG@20":>9}')
print('-' * 63)
print(f'{"BM25_original (Assignment 2)":<35} {0.212:>6.4f} {0.1752:>8.4f} {0.3570:>9.4f}')
print(f'{"LambdaMart (14 features)":<35} {agg[P@10]:>6.4f} {agg[MAP]:>8.4f} {agg[nDCG@20]:>9.4f}')
print(f'{"LambdaMart (BM25 components)":<35} {agg2[P@10]:>6.4f} {agg2[MAP]:>8.4f} {agg2[nDCG@20]:>9.4f}')


=== Final Comparison (all methods vs Assignment 2 baseline) ===
Method                                P@10      MAP   nDCG@20
---------------------------------------------------------------
BM25_original (Assignment 2)        0.2120   0.1752    0.3570
LambdaMart (14 features)            0.1480   0.1414    0.2678
LambdaMart (BM25 components)        0.1200   0.1180    0.2355
